# Google Books API — Data Retrieval

Second data source for the Book Recommendation System.  
Queries the Google Books Volumes API by subject, mirroring the 8 GoodReads Listopia categories scraped in `Webscraping_books.ipynb`.  
Output schema is aligned to `goodreads.csv` for a clean merge downstream.

instructions on Google Books API available here: https://developers.google.com/books/docs/v1/reference/?apix=true

## 1. Imports & Config

In [26]:
import os
import time
import logging
import requests
import pandas as pd
import yaml
from dotenv import load_dotenv
import sys
sys.path.insert(0, '.')
from functions import out_csv

# Load API key from .env
load_dotenv(override=True)
API_KEY = os.getenv('ggl_books_api')
assert API_KEY, 'API key not found — check .env for ggl_books_api'

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
log = logging.getLogger(__name__)

YAML_PATH = '../config.yaml'

print(f'API key loaded: {API_KEY[:6]}...{API_KEY[-4:]}')

API key loaded: AIzaSy...yJ1A


## 2. Query Configuration

Each entry maps a human-readable list name (matching GoodReads `source_list` format)  
to a Google Books subject query term.  
Target: ~50 books per list × 8 lists = ~320 records (before deduplication).

In [42]:
# Maps source_list label → Google Books subject query string
LISTS = {
    'Best Books Ever':             'subject:fiction',
    'Books You Should Read':       'subject:classics',
    'Young Adult':                 'subject:"young adult"',
    'Books That Should Be Movies': 'subject:adventure',
    '20th Century Literature':     'subject:fiction',
    'Best Book Boyfriends':        'subject:romance',
    'Best Epic Fantasy':           'subject:fantasy',
    'Best Historical Fiction':     'subject:"historical fiction"',
}

# Google Books API settings
BASE_URL    = 'https://www.googleapis.com/books/v1/volumes'
MAX_RESULTS = 40   # API hard cap per call
TARGET_PER_LIST = 70  # books to retrieve per list
DELAY_SECONDS = 1.0   # polite delay between calls

## 3. Field Extraction Helper

Extracts all required fields from a single `volumeInfo` dict.  
Returns `None` for any missing field — downstream cleaning handles nulls.  
Schema mirrors `goodreads.csv`: `book_name`, `book_url`, `source_list`, `author`,  
`rating`, `synopsis`, `pages`, `pub_date`, `genres`, `img_url`.

In [43]:
def extract_fields(item: dict, source_list: str) -> dict:
    """
    Parse a single Google Books API volume item into the project schema.

    Parameters
    ----------
    item : dict
        A single element from the API response 'items' list.
    source_list : str
        The list label to inject (mirrors GoodReads source_list values).

    Returns
    -------
    dict with keys matching goodreads.csv column names.
    Missing fields are None — not dropped here.
    """
    vi = item.get('volumeInfo', {})   #if it's missing, retunrs an empty dict instead of None so next get do not crash

    # authors: take first only, matching GoodReads single-author format
    authors = vi.get('authors', [])
    author  = authors[0] if authors else None

    # genres: pipe-separated string, matching GoodReads format
    categories = vi.get('categories', [])
    genres = '|'.join(categories) if categories else None

    # thumbnail image
    img_links = vi.get('imageLinks', {})
    img_url   = img_links.get('thumbnail') or img_links.get('smallThumbnail')

    return {
        'book_name'   : vi.get('title'),
        'book_url'    : vi.get('infoLink'),        # Google Books page (no list URL equivalent)
        'source_list' : source_list,
        'author'      : author,
        'rating'      : vi.get('averageRating'),   # float or None
        'synopsis'    : vi.get('description'),
        'pages'       : vi.get('pageCount'),       # int or None
        'pub_date'    : vi.get('publishedDate'),   # string, variable precision: YYYY / YYYY-MM / YYYY-MM-DD
        'genres'      : genres,
        'img_url'     : img_url,
        'data_source' : 'google_books',
    }

## 4. Fetcher Function

Queries the API for one list label, paginates up to `target` books,  
and returns a list of extracted record dicts.  
Implements exponential backoff on HTTP 429 (rate limit) and retries on 5xx errors.

In [46]:
def fetch_list(list_name: str, query: str, target: int = TARGET_PER_LIST) -> list:
    """
    Fetch up to `target` books for one list from the Google Books API.

    Pagination: startIndex increments by MAX_RESULTS each call.
    Stops early if the API returns fewer items than requested (end of results).
    Retries up to 3 times on 5xx errors with exponential backoff.
    Backs off 60 s on 429 (quota exceeded).

    Parameters
    ----------
    list_name : str   Human-readable label injected as source_list.
    query     : str   Subject search string sent to the API.
    target    : int   Maximum books to retrieve for this list.

    Returns
    -------
    list of dicts (one per book, schema from extract_fields)
    """
    records     = []
    start_index = 0

    while len(records) < target:
        params = {
            'q'          : query,
            'maxResults' : MAX_RESULTS,
            'startIndex' : start_index,
            'printType'  : 'books',
            'langRestrict': 'en',
            'key'        : API_KEY,
        }

        # Retry loop for transient server errors
        for attempt in range(3):
            try:
                resp = requests.get(BASE_URL, params=params, timeout=10)

                if resp.status_code == 429:
                    log.warning('Rate limit hit — sleeping 60 s')
                    time.sleep(60)
                    continue

                if resp.status_code >= 500:
                    wait = 2 ** attempt
                    log.warning(f'Server error {resp.status_code} — retry {attempt+1}/3 in {wait}s')
                    time.sleep(wait)
                    continue

                resp.raise_for_status()
                break  # success

            except requests.exceptions.RequestException as e:
                log.error(f'Request failed: {e}')
                if attempt == 2:
                    log.error(f'Giving up on list "{list_name}" at startIndex={start_index}')
                    return records
                time.sleep(2 ** attempt)

        data  = resp.json()
        items = data.get('items', [])

        if not items:
            log.info(f'[{list_name}] No more results at startIndex={start_index}')
            break

        for item in items:
            records.append(extract_fields(item, list_name))
            if len(records) >= target:
                break

        log.info(f'[{list_name}] Fetched {len(records)} / {target}')

        # Stop if API returned fewer than requested (end of result set)
        start_index += len(items) 

        start_index += MAX_RESULTS
        time.sleep(DELAY_SECONDS)

    return records

## 5. Pilot Run — Measure Rating Coverage

Fetch 40 books from the first list only.  
Goal: measure `averageRating` null-rate before committing to full run.  
Decision rule: if null-rate > 60%, the rating column is too sparse to be useful — flag and decide.

In [47]:
#testing API 503 error
params = {
    'q'           : 'classics fiction',
    'maxResults'  : 40,
    'startIndex'  : 0,
    'printType'   : 'books',
    'langRestrict': 'en',
    'key'         : API_KEY,
}
resp = requests.get(BASE_URL, params=params, timeout=10)
data = resp.json()
print(resp.status_code)
print('totalItems:', data.get('totalItems'))
print('items returned:', len(data.get('items', [])))

200
totalItems: 300
items returned: 20


In [48]:
# Pilot: first list only
pilot_list_name  = list(LISTS.keys())[0]
pilot_query      = LISTS[pilot_list_name]

log.info(f'Pilot run: "{pilot_list_name}" → query: "{pilot_query}"')
pilot_records = fetch_list(pilot_list_name, pilot_query, target=40)

pilot_df = pd.DataFrame(pilot_records)

# Rating null-rate
null_rate = pilot_df['rating'].isna().mean()
print(f'\nPilot results: {len(pilot_df)} books')
print(f'Rating null-rate: {null_rate:.1%}')
print(f'\nSample (5 rows):')
pilot_df[['book_name', 'author', 'rating', 'genres', 'pages', 'pub_date']].head()

2026-07-08 18:02:12,709 | INFO | Pilot run: "Best Books Ever" → query: "subject:fiction"
2026-07-08 18:02:13,243 | INFO | [Best Books Ever] Fetched 20 / 40
2026-07-08 18:02:15,216 | INFO | [Best Books Ever] Fetched 40 / 40



Pilot results: 40 books
Rating null-rate: 95.0%

Sample (5 rows):


,book_name,author,rating,genres,pages,pub_date
0,Persuasion,Jane Austen,NaN,Fiction,260,2026-01-01
1,The Face of Another,Kōbō Abe,NaN,Fiction,260,1992
2,Oliver Twist,Charles Dickens,NaN,Fiction,220,2019-01-01
3,The Happy Prince and Other Stories,Oscar Wilde,NaN,Fiction,105,2019-06-28
4,Sense and Sensibility - Illustrated Edition,Jane Austen,NaN,Fiction,579,2017-10-23


## 6. Full Fetch — All 8 Lists

Run only after reviewing pilot results above.  
Iterates all lists, collects records, concatenates into a single DataFrame.

In [49]:
all_records = []

for list_name, query in LISTS.items():
    log.info(f'--- Starting list: {list_name} ---')
    records = fetch_list(list_name, query, target=TARGET_PER_LIST)
    all_records.extend(records)
    log.info(f'List complete: {len(records)} books retrieved')
    time.sleep(DELAY_SECONDS)

google_books_df = pd.DataFrame(all_records)
print(f'\nTotal records fetched: {len(google_books_df)}')
google_books_df.head(3)

2026-07-08 18:02:45,515 | INFO | --- Starting list: Best Books Ever ---
2026-07-08 18:02:46,912 | INFO | [Best Books Ever] Fetched 20 / 70
2026-07-08 18:02:48,209 | WARNING | Server error 503 — retry 1/3 in 1s
2026-07-08 18:02:49,781 | INFO | [Best Books Ever] Fetched 40 / 70
2026-07-08 18:02:51,465 | INFO | [Best Books Ever] Fetched 60 / 70
2026-07-08 18:02:53,129 | INFO | [Best Books Ever] Fetched 70 / 70
2026-07-08 18:02:54,138 | INFO | List complete: 70 books retrieved
2026-07-08 18:02:55,141 | INFO | --- Starting list: Books You Should Read ---
2026-07-08 18:02:55,511 | WARNING | Server error 503 — retry 1/3 in 1s
2026-07-08 18:02:56,845 | WARNING | Server error 503 — retry 2/3 in 2s
2026-07-08 18:02:59,743 | INFO | [Books You Should Read] Fetched 20 / 70
2026-07-08 18:03:01,351 | INFO | [Books You Should Read] Fetched 40 / 70
2026-07-08 18:03:02,991 | INFO | [Books You Should Read] Fetched 60 / 70
2026-07-08 18:03:04,936 | INFO | [Books You Should Read] Fetched 70 / 70
2026-07-08


Total records fetched: 480


,book_name,book_url,source_list,author,rating,synopsis,pages,pub_date,genres,img_url,data_source
0,Persuasion,https://play.google.com/store/books/details?id...,Best Books Ever,Jane Austen,NaN,Discover the timeless allure of love and secon...,260.0,2026-01-01,Fiction,http://books.google.com/books/content?id=UrrTE...,google_books
1,The Face of Another,http://books.google.de/books?id=_DYNAQAAMAAJ&d...,Best Books Ever,Kōbō Abe,NaN,The Japanese novelist Kobo Abe has often been ...,260.0,1992,Fiction,NaN,google_books
2,Oliver Twist,https://play.google.com/store/books/details?id...,Best Books Ever,Charles Dickens,NaN,JAICO ILLUSTARTED CLASSICS SERIES is a collect...,220.0,2019-01-01,Fiction,http://books.google.com/books/content?id=3oSqD...,google_books


## 7. Basic Validation & Deduplication

Check shape, null counts, and deduplicate on `book_name` + `author`.  
This is a sanity check only — full cleaning happens in the cleaning notebook.

In [50]:
print('=== Shape before dedup ===')
print(google_books_df.shape)

print('\n=== Null counts ===')
print(google_books_df.isna().sum())

print('\n=== Rating distribution ===')
print(google_books_df['rating'].describe())

# Deduplicate: same book can appear across multiple list queries
before = len(google_books_df)
google_books_df = google_books_df.drop_duplicates(subset=['book_name', 'author'])
after = len(google_books_df)
print(f'\nDuplicates removed: {before - after} | Remaining: {after}')

=== Shape before dedup ===
(480, 11)

=== Null counts ===
book_name        0
book_url         0
source_list      0
author           4
rating         434
synopsis        62
pages           16
pub_date         4
genres          94
img_url         96
data_source      0
dtype: int64

=== Rating distribution ===
count    46.000000
mean      4.282609
std       0.898469
min       1.000000
25%       4.000000
50%       4.500000
75%       5.000000
max       5.000000
Name: rating, dtype: float64

Duplicates removed: 86 | Remaining: 394


## 8. Save to Raw CSV

Saves to `data/raw/google_books.csv` via `out_csv()` from `functions.py`.  
Requires `config.yaml` to have the output path defined under the correct key.

In [51]:
out_csv(
    df                  = google_books_df,
    yaml_path           = YAML_PATH,
    output_section_yaml = 'raw_data',
    file_name           = 'raw_api'
)

print(f'Saved {len(google_books_df)} records.')

File saved to: ../data/raw/google_books.csv
Saved 394 records.
